# H2 — Rótulos humanos × rótulos de LLM (Colab gratuito, GPU T4)

**Antes de rodar:** Ambiente de execução → Alterar o tipo → **GPU T4**. Depois: **Ambiente de execução → Executar tudo**.

O que este notebook faz (sem custo; o Qwen2.5-7B é um modelo aberto, baixado do Hugging Face):
1. O **Qwen2.5-7B-Instruct** (4 bits) rotula os mesmos textos que os humanos do RePro rotularam: primeiro a validação (599, para medir a concordância e o tempo) e depois o treino (5.589).
2. O **BERTimbau** é treinado com esses rótulos (3 sementes) e avaliado no mesmo teste humano.
3. Tudo fica salvo no Google Drive. **Se a sessão cair ou a cota acabar, abra de novo e clique em Executar tudo: continua de onde parou.**
4. No fim, baixa `resultados_h2.zip`.

In [ ]:
# 1) Repositório, pacote e Drive (cache do LLM no Drive para retomar)
import os, sys
from pathlib import Path
if "google.colab" in sys.modules:
    !git clone -q https://github.com/vnevesl/pln-absa-olist.git 2>/dev/null || git -C pln-absa-olist pull -q
    %cd /content/pln-absa-olist
    !pip -q install -e ".[llm]"
    from google.colab import drive
    try:
        drive.mount("/content/drive")
        BASE = Path("/content/drive/MyDrive/pln-absa-olist")
    except Exception as erro:
        print(f"⚠️ Drive não conectou ({erro}). NÃO feche a aba até baixar o zip no fim.")
        BASE = Path("/content/pln-absa-olist-saida")
else:
    BASE = Path("results/h2_local")
_src = str(Path("src").resolve())
if _src not in sys.path:
    sys.path.insert(0, _src)
SAIDA = BASE / "resultados_h2"; SAIDA.mkdir(parents=True, exist_ok=True)
CACHE = BASE / "llm_cache"; CACHE.mkdir(parents=True, exist_ok=True)
Path("data").mkdir(exist_ok=True)
if not Path("data/llm_cache").exists():
    os.symlink(CACHE, "data/llm_cache")  # respostas do LLM vão direto para o Drive
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NENHUMA — troque o ambiente para GPU T4!")

In [ ]:
# 2) Dados: RePro, validação dividida em exemplos do prompt (demo) e avaliação (val_dev)
import json, time
import pandas as pd
from sklearn.model_selection import train_test_split
from absa.config import ASPECTS, LABEL_COLUMNS, SEED
from absa.llm_labeler import hf_generator, label_texts, select_fixed_examples
from absa.metrics import bootstrap_ci, per_aspect_report, tuple_f1
from absa.repro import load_repro

treino, val, teste, _ = load_repro()
demo, val_dev = train_test_split(val, train_size=200, stratify=val.nota, random_state=SEED)
demo, val_dev = demo.reset_index(drop=True), val_dev.reset_index(drop=True)
exemplos = select_fixed_examples(demo, k=6)
MODELO = "Qwen/Qwen2.5-7B-Instruct"
print(f"treino {len(treino)} · val_dev {len(val_dev)} · {len(exemplos)} exemplos no prompt")

In [ ]:
# 3) Carrega o Qwen2.5-7B em 4 bits (~5 GB; o download leva alguns minutos)
ja_rotulado = all((SAIDA / f"repro_{n}_qwen7b.csv").exists() for n in ["valdev", "treino"])
generate = None if ja_rotulado else hf_generator(MODELO, load_in_4bit=True, max_new_tokens=60)
print("rótulos já existem no Drive — pulando o LLM" if ja_rotulado else "Qwen carregado")

In [ ]:
# 4) Rotula a validação (599) e mede a concordância LLM × humano e o tempo
def sem_invalidos(rot):
    pred = rot[LABEL_COLUMNS].copy()
    pred.loc[~rot.valido.to_numpy(), LABEL_COLUMNS] = "ausente"  # inválido conta como erro
    return pred.fillna("ausente")

def rotular(df, nome, lote=16):
    arq = SAIDA / f"repro_{nome}_qwen7b.csv"
    if arq.exists():
        print(f"✓ {nome} já rotulado — lendo do Drive"); return pd.read_csv(arq)
    t0 = time.time()
    rot, _ = label_texts(df.texto_modelo.tolist(), generate, MODELO, exemplos,
                         cache_name=f"qwen7b_{nome}", batch_size=lote)
    out = df[["review_id"]].copy()
    for a in ASPECTS:
        out[f"asp_{a}"] = [r[a] if r else None for r in rot]
    out["valido"] = [r is not None for r in rot]
    out.to_csv(arq, index=False)
    print(f"{nome}: {out.valido.sum()}/{len(out)} respostas válidas ({(time.time() - t0) / 60:.1f} min)")
    return out

t0 = time.time()
rot_val = rotular(val_dev, "valdev")
seg_por_texto = (time.time() - t0) / len(val_dev)
pred = sem_invalidos(rot_val)
f1, lo, hi = bootstrap_ci(val_dev, pred)
print(f"\nConcordância LLM × humano na validação: F1 de pares {f1:.3f} (IC95% {lo:.3f}–{hi:.3f})")
display(per_aspect_report(val_dev, pred).round(3))
print(f"Estimativa para o treino: ~{seg_por_texto * len(treino) / 60:.0f} min (se ainda não estiver em cache)")

In [ ]:
# 5) Rotula o treino (5.589) — a parte mais longa; retomável pelo cache no Drive
rot_treino = rotular(treino, "treino")
ok = rot_treino.valido.to_numpy()
pred_t = sem_invalidos(rot_treino)
qual = {"f1_pares_valdev": f1, "ic_valdev": [lo, hi], "f1_pares_treino": tuple_f1(treino, pred_t),
        "validos_treino": int(ok.sum()), "seg_por_texto": seg_por_texto, "modelo": MODELO,
        "exemplos_prompt": len(exemplos)}
json.dump(qual, open(SAIDA / "qualidade_rotulos_qwen7b.json", "w"), indent=1, default=float)
print(f"Rótulos do LLM no treino: F1 de pares contra o humano = {qual['f1_pares_treino']:.3f}")
del generate; import gc; gc.collect(); torch.cuda.empty_cache()  # libera a GPU para o BERTimbau

In [ ]:
# 6) BERTimbau treinado com os rótulos do LLM (3 sementes), avaliado no teste humano
from absa.config import POLARITIES
from absa.models.encoder import ENCODERS, EncoderAspectClassifier, TrainConfig
from absa.preprocess import clean_light

entrada = lambda df: df.texto_modelo.map(clean_light).tolist()
treino_llm = treino[["review_id", "texto_modelo"]].merge(rot_treino[rot_treino.valido], on="review_id")
CKPT = Path("/content/ckpt" if "google.colab" in sys.modules else SAIDA / "ckpt"); CKPT.mkdir(parents=True, exist_ok=True)
print(f"treino com rótulos do LLM: {len(treino_llm)} textos")

for seed in [13, 42, 87]:
    arq = SAIDA / f"pred_teste_bertimbau_L_s{seed}.csv"
    if arq.exists():
        print(f"✓ L s{seed} já feito — pulando"); continue
    t0 = time.time()
    clf = EncoderAspectClassifier(TrainConfig(model_name=ENCODERS["bertimbau"], epochs=4, lr=3e-5,
                                              batch_size=32, max_len=128, seed=seed))
    ckpt = CKPT / f"L_s{seed}.pt"
    clf.fit(entrada(treino_llm), treino_llm, entrada(val_dev), val_dev, save_path=ckpt)  # época escolhida no humano
    clf.load(ckpt)
    proba = clf.predict_proba(entrada(teste))
    p = pd.DataFrame({c: [POLARITIES[i] for i in proba[:, j].argmax(1)] for j, c in enumerate(LABEL_COLUMNS)})
    p.insert(0, "review_id", teste.review_id.values)
    p.to_csv(arq, index=False)
    json.dump({"cond": "L", "seed": seed, "n_treino": len(treino_llm), "minutos": (time.time() - t0) / 60,
               "historico": clf.history}, open(SAIDA / f"log_bertimbau_L_s{seed}.json", "w"), indent=1)
    print(f"→ L s{seed}: F1 de pares no teste {tuple_f1(teste, p):.3f}")
    ckpt.unlink(missing_ok=True); del clf; torch.cuda.empty_cache()

In [ ]:
# 7) Baixar os resultados
import shutil
shutil.make_archive("/content/resultados_h2", "zip", SAIDA)
if "google.colab" in sys.modules:
    from google.colab import files
    files.download("/content/resultados_h2.zip")